In [1]:
from flax import nnx
import pandas as pd
import glob
from pathlib import Path
import json
import os
import sys
sys.path.append("..")

from qnrl.algorithms.algorithm import Algorithm

import wandb
api = wandb.Api()

wandb: Currently logged in as: zanderman (zanderman-virginia-tech) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [2]:
root_sweeps = Path("../wandb-sweep-data/")
root_sweeps.mkdir(parents=True, exist_ok=True)

In [3]:
project_id = "qnrl"

In [4]:
import ast
from qnrl.serialization import hash_serializable_object
def get_run_config_dict_from_metadata(metadata_file: Path):
    """Compute run config hash from original run program command string."""
    # Load metadata as JSON dictionary.
    with open(str(metadata_file), 'r') as f:
        metadata = json.load(f)
    
    # Get config string from run program command arguments.
    # Format is: --config="{...}"
    cfg_str = metadata['args'][1].split("--config=")[1]
    # Convert this string to dictionary via literal evaluation.
    cfg_dict = ast.literal_eval(cfg_str)
    return cfg_dict

In [5]:
# Load algorithm config for sweep and run.
sweep_id = "1nlfftsy"
run_id = "5ti67g7h"
dir_run = root_sweeps/f"sweep-{sweep_id}"/f"run-{run_id}"

cfg = get_run_config_dict_from_metadata(dir_run/"wandb-metadata.json")
alg = Algorithm.from_dict(cfg)

nnx.display(alg)

In [6]:
# Display initial state of model parameters.
nnx.display(nnx.state(alg.model))

In [7]:
# Load parameters from model save.
model_dirs = {Path(root).stem: Path(root) for root in glob.glob(str(dir_run/"models"/"*"))}
alg.load(model_dirs)
nnx.display(nnx.state(alg.model))